# Transformar bases de PIB para formato long

Este notebook transforma as planilhas `PIB mesos.xlsx` e `PIB municipios.xlsx` para formato long, com uma linha por unidade geografica e data de referencia.

As planilhas originais devem estar em `Dados/Bruto`, e as planilhas processadas serao salvas em `Dados/Processado`.

A coluna `Tipo PIB` indica se o valor e `Observado` ou `Projeção`. Por regra, anos ate 2023 sao observados e anos a partir de 2024 sao projecoes.

In [1]:
from pathlib import Path

import pandas as pd

PASTA_ATUAL = Path.cwd()
PASTA_PROJETO = PASTA_ATUAL.parent if PASTA_ATUAL.name == "Notebooks" else PASTA_ATUAL
PASTA_DADOS_BRUTOS = PASTA_PROJETO / "Dados" / "Bruto"
PASTA_DADOS_PROCESSADOS = PASTA_PROJETO / "Dados" / "Processado"

ARQUIVO_MESOS = PASTA_DADOS_BRUTOS / "PIB mesos.xlsx"
ARQUIVO_MUNICIPIOS = PASTA_DADOS_BRUTOS / "PIB municipios.xlsx"

ANO_INICIO_PROJECAO = 2024
CORRECOES_MESORREGIAO = {
    "421935": "Vale do Itajai",  # Vitor Meireles
    "421690": "Oeste Catarinense",  # São Lourenço do Oeste
}

## Funcao de transformacao

A funcao abaixo preserva as colunas de identificacao da base, transforma as colunas de ano em linhas e cria as colunas `Data de Referência`, `PIB` e `Tipo PIB`.

In [2]:
def transformar_para_long(df: pd.DataFrame, colunas_id: list[str]) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]

    colunas_ano = [col for col in df.columns if col.isdigit()]
    colunas_ano = sorted(colunas_ano, key=lambda col: int(col))

    long = df.melt(
        id_vars=colunas_id,
        value_vars=colunas_ano,
        var_name="Ano",
        value_name="PIB",
    )

    long["Ano"] = long["Ano"].astype(int)
    long["Data de Referência"] = pd.to_datetime(
        long["Ano"].astype(str) + "-01-01",
        format="%Y-%m-%d",
    )
    long["Tipo PIB"] = long["Ano"].map(
        lambda ano: "Projeção" if ano >= ANO_INICIO_PROJECAO else "Observado"
    )

    colunas_saida = ["Data de Referência", *colunas_id, "Tipo PIB", "PIB"]
    return long[colunas_saida].sort_values(colunas_id + ["Data de Referência"]).reset_index(drop=True)


def aplicar_correcoes_mesorregiao(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    codigos = df["Cód. Munic"].astype(str).str.replace(r"\D", "", regex=True)

    for codigo, mesorregiao in CORRECOES_MESORREGIAO.items():
        df.loc[codigos == codigo, "Mesorregião"] = mesorregiao

    return df


def recalcular_mesos_long(pib_municipios_long: pd.DataFrame) -> pd.DataFrame:
    municipios = pib_municipios_long[pib_municipios_long["Município"] != "-"]
    mesos = (
        municipios.groupby(["Data de Referência", "Mesorregião", "Tipo PIB"], as_index=False)["PIB"]
        .sum()
        .rename(columns={"Mesorregião": "Mesos"})
    )

    santa_catarina = (
        pib_municipios_long[pib_municipios_long["Município"] == "-"]
        [["Data de Referência", "Tipo PIB", "PIB"]]
        .assign(Mesos="Santa Catarina")
    )

    return (
        pd.concat([mesos, santa_catarina], ignore_index=True)
        [["Data de Referência", "Mesos", "Tipo PIB", "PIB"]]
        .sort_values(["Mesos", "Data de Referência"])
        .reset_index(drop=True)
    )

## Ler e transformar as bases

In [3]:
pib_mesos = pd.read_excel(ARQUIVO_MESOS)
pib_municipios = aplicar_correcoes_mesorregiao(pd.read_excel(ARQUIVO_MUNICIPIOS))


pib_municipios_long = transformar_para_long(
    pib_municipios,
    colunas_id=["Cód. Munic", "Município", "Mesorregião", "VPs"],
)

pib_mesos_long = recalcular_mesos_long(pib_municipios_long)

pib_mesos_long.head(), pib_municipios_long.head()

(  Data de Referência                 Mesos   Tipo PIB           PIB
 0         2002-01-01  Grande Florianópolis  Observado  4.464116e+10
 1         2003-01-01  Grande Florianópolis  Observado  4.466393e+10
 2         2004-01-01  Grande Florianópolis  Observado  4.590022e+10
 3         2005-01-01  Grande Florianópolis  Observado  4.999905e+10
 4         2006-01-01  Grande Florianópolis  Observado  5.182016e+10,
   Data de Referência Cód. Munic Município Mesorregião             VPs  \
 0         2002-01-01          -         -           -  Santa Catarina   
 1         2003-01-01          -         -           -  Santa Catarina   
 2         2004-01-01          -         -           -  Santa Catarina   
 3         2005-01-01          -         -           -  Santa Catarina   
 4         2006-01-01          -         -           -  Santa Catarina   
 
     Tipo PIB           PIB  
 0  Observado  3.024251e+11  
 1  Observado  3.105588e+11  
 2  Observado  3.310055e+11  
 3  Observado  3.41

## Conferencias rapidas

In [4]:
print("PIB mesos - formato original:", pib_mesos.shape)
print("PIB mesos - formato long:", pib_mesos_long.shape)
print()
print("PIB municipios - formato original:", pib_municipios.shape)
print("PIB municipios - formato long:", pib_municipios_long.shape)
print()
print("Tipos na base de municipios:")
print(pib_municipios_long["Tipo PIB"].value_counts())

pib_municipios_long.query("Município == 'Abdon Batista'").head()

PIB mesos - formato original: (7, 30)
PIB mesos - formato long: (203, 4)

PIB municipios - formato original: (296, 33)
PIB municipios - formato long: (8584, 7)

Tipos na base de municipios:
Tipo PIB
Observado    6512
Projeção     2072
Name: count, dtype: int64


,Data de Referência,Cód. Munic,Município,Mesorregião,VPs,Tipo PIB,PIB
29,2002-01-01,420005,Abdon Batista,Serrana,Centro-Oeste,Observado,83819032.0
30,2003-01-01,420005,Abdon Batista,Serrana,Centro-Oeste,Observado,84264681.0
31,2004-01-01,420005,Abdon Batista,Serrana,Centro-Oeste,Observado,72946333.0
32,2005-01-01,420005,Abdon Batista,Serrana,Centro-Oeste,Observado,64553554.0
33,2006-01-01,420005,Abdon Batista,Serrana,Centro-Oeste,Observado,71056167.0


## Salvar arquivos transformados

Os arquivos abaixo serao salvos em `Dados/Processado`.

In [5]:
PASTA_DADOS_PROCESSADOS.mkdir(parents=True, exist_ok=True)

SAIDA_MESOS_XLSX = PASTA_DADOS_PROCESSADOS / "PIB mesos long.xlsx"
SAIDA_MUNICIPIOS_XLSX = PASTA_DADOS_PROCESSADOS / "PIB municipios long.xlsx"

pib_mesos_long.to_excel(SAIDA_MESOS_XLSX, index=False)
pib_municipios_long.to_excel(SAIDA_MUNICIPIOS_XLSX, index=False)

print("Arquivos salvos em Excel:")
print(SAIDA_MESOS_XLSX)
print(SAIDA_MUNICIPIOS_XLSX)

Arquivos salvos em Excel:
c:\Users\joao-giorio\Desktop\Painel PIB Municípios\Dados\Processado\PIB mesos long.xlsx
c:\Users\joao-giorio\Desktop\Painel PIB Municípios\Dados\Processado\PIB municipios long.xlsx
